In [7]:
import os
import time
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

# ======================
# 1. 读取所有股票数据
# ======================
df_list = []
for file in os.listdir("data/stock"):
    code = file.split("_")[1].replace(".csv", "")
    df = pd.read_csv(f"data/stock/{file}", parse_dates=["日期"])
    df["code"] = code
    df_list.append(df)
df = pd.concat(df_list, ignore_index=True)

# ======================
# 2. 数据清洗（作业6项）
# ======================
df.rename(columns={"日期": "date"}, inplace=True)

# 1. 缺失值统计
print("【清洗步骤1】缺失值统计")
print(df.isnull().sum())

# 2. 排序 + 分组填充（修复警告）
df = df.sort_values(by=["code", "date"]).reset_index(drop=True)
df = df.groupby("code", group_keys=False).apply(lambda x: x.ffill()).reset_index(drop=True)

# 3. 重复值处理
before_dup = df.shape[0]
df = df.drop_duplicates(subset=["code", "date"])
print(f"【清洗步骤2】删除重复行数: {before_dup - df.shape[0]}")

# 4. 日期格式统一
df["date"] = pd.to_datetime(df["date"])

# 5. 计算对数收益率（修复报错）
def calc_ret(x):
    x = x.copy()
    return np.log(x / x.shift(1))

df["log_return"] = df.groupby("code", group_keys=False)["收盘"].apply(calc_ret)

# 6. 极端值标注
df["is_extreme"] = df["log_return"].abs() > np.log(1.2)

# 保存清洗后CSV
df.to_csv("data/clean/stock_clean.csv", index=False, encoding="utf-8-sig")

# ======================
# 3. 宽表 ↔ 长表转换
# ======================
df_wide = df.pivot(index="date", columns="code", values="收盘")
df_long = df_wide.reset_index().melt(id_vars="date", var_name="code", value_name="close")
print("【清洗步骤3】宽表适合对比分析，长表适合回归建模")

# ======================
# 4. CSV VS Parquet 对比
# ======================
df.to_parquet("data/clean/stock_clean.parquet", index=False)

# 读取速度对比
t0 = time.time()
pd.read_csv("data/clean/stock_clean.csv")
t_csv = time.time() - t0

t0 = time.time()
pd.read_parquet("data/clean/stock_clean.parquet")
t_pq = time.time() - t0

size_csv = os.path.getsize("data/clean/stock_clean.csv") / 1024
size_pq = os.path.getsize("data/clean/stock_clean.parquet") / 1024

print(f"CSV  读取时间: {t_csv:.2f}s, 文件大小: {size_csv:.0f}KB")
print(f"PARQ 读取时间: {t_pq:.2f}s, 文件大小: {size_pq:.0f}KB")
print("【清洗步骤4】Parquet 体积更小、读取更快")

# ======================
# 5. 合并指数 + 宏观数据
# ======================
hs300 = pd.read_csv("data/index/index_hs300.csv", parse_dates=["日期"])
hs300["mkt_return"] = np.log(hs300["收盘"] / hs300["收盘"].shift(1))
df = df.merge(hs300[["日期", "mkt_return"]], left_on="date", right_on="日期", how="left")

cpi = pd.read_csv("data/macro/macro_cpi.csv", parse_dates=["月份"])
cpi["month"] = cpi["月份"].dt.to_period("M")
df["month"] = df["date"].dt.to_period("M")
df = df.merge(cpi[["month", "同比增长"]], on="month", how="left")

df.to_csv("data/combined/combined_data.csv", index=False, encoding="utf-8-sig")
print("✅ 02_clean 全部完成！")

【清洗步骤1】缺失值统计
date    0
开盘      0
收盘      0
最高      0
最低      0
成交量     0
成交额     0
code    0
dtype: int64
【清洗步骤2】删除重复行数: 0


C:\Users\sxl\AppData\Local\Temp\ipykernel_25952\1972170128.py:29: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("code", group_keys=False).apply(lambda x: x.ffill()).reset_index(drop=True)
C:\Users\sxl\anaconda3\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


【清洗步骤3】宽表适合对比分析，长表适合回归建模
CSV  读取时间: 0.04s, 文件大小: 2213KB
PARQ 读取时间: 0.05s, 文件大小: 1035KB
【清洗步骤4】Parquet 体积更小、读取更快
✅ 02_clean 全部完成！
